In [11]:
import torch.nn as nn
import math
from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import load_dataset
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling
 
MODEL_NAME = "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T"
RANK       = 128
LEVEL      = 4
MIN_DIM    = 64
BLOCK_SIZE = 512
 
# We do NOT set a single DEVICE — model will span both GPUs via device_map
print(f"rank={RANK}  level={LEVEL}  min_dim={MIN_DIM}")
 

rank=128  level=4  min_dim=64


In [12]:
# ────────────────────────────────────────────────────────────
# CELL 4 — Extract Wq, Wk, Wv  (CPU, fp16 → fp32)
#
#  Load on CPU in fp16 (only ~2.2 GB).  Extract weights as
#  fp32.  Delete base model immediately after — free memory
#  before the SVD step.
# ────────────────────────────────────────────────────────────
 
print("Loading model on CPU (fp16) for weight extraction …")
_base = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="cpu",
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token
 
weight_dict = {}
for i, layer in enumerate(_base.model.layers):
    attn = layer.self_attn
    for name, proj in [("q", attn.q_proj), ("k", attn.k_proj), ("v", attn.v_proj)]:
        weight_dict[f"layer{i}.{name}"] = proj.weight.detach().float().cpu()
    print(f"  layer {i:2d} extracted")
 
n_layers = len(_base.model.layers)          # 22 for TinyLlama
del _base
torch.cuda.empty_cache()
import gc; gc.collect()
print(f"\n{len(weight_dict)} matrices extracted.  Base model freed.")

Loading model on CPU (fp16) for weight extraction …


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

  layer  0 extracted
  layer  1 extracted
  layer  2 extracted
  layer  3 extracted
  layer  4 extracted
  layer  5 extracted
  layer  6 extracted
  layer  7 extracted
  layer  8 extracted
  layer  9 extracted
  layer 10 extracted
  layer 11 extracted
  layer 12 extracted
  layer 13 extracted
  layer 14 extracted
  layer 15 extracted
  layer 16 extracted
  layer 17 extracted
  layer 18 extracted
  layer 19 extracted
  layer 20 extracted
  layer 21 extracted

66 matrices extracted.  Base model freed.


In [13]:
def truncated_svd(W: torch.Tensor, rank: int):
    """Economy SVD, truncated to rank.  Returns U(out,r), S(r,), V(r,in)."""
    U, S, Vh = torch.linalg.svd(W, full_matrices=False)
    r = min(rank, S.shape[0])
    return U[:, :r].clone(), S[:r].clone(), Vh[:r, :].clone()
 
 
def build_param_node(W: torch.Tensor, rank: int, level: int, min_dim: int) -> dict:
    """
    Recursively decompose W into a tree of plain dicts.
    Every node — leaf or internal — stores (U, S, V) tensors.
    All tensors float32 on CPU.
    """
    out_dim, in_dim = W.shape
 
    if level == 0 or min(out_dim, in_dim) <= min_dim:
        U, S, V = truncated_svd(W, rank)
        return {"is_leaf": True, "U": U, "S": S, "V": V}
 
    in_mid  = in_dim  // 2
    out_mid = out_dim // 2
 
    W11 = W[:out_mid,  :in_mid ]
    W12 = W[:out_mid,  in_mid: ]
    W21 = W[out_mid:,  :in_mid ]
    W22 = W[out_mid:,  in_mid: ]
 
    U12, S12, V12 = truncated_svd(W12, rank)
    U21, S21, V21 = truncated_svd(W21, rank)
 
    return {
        "is_leaf" : False,
        "in_mid"  : in_mid,
        "out_mid" : out_mid,
        "A11"     : build_param_node(W11, rank, level - 1, min_dim),
        "A22"     : build_param_node(W22, rank, level - 1, min_dim),
        "A12"     : {"U": U12, "S": S12, "V": V12},
        "A21"     : {"U": U21, "S": S21, "V": V21},
    }
 
 
print("Running offline SVD …")
param_nodes = {}
for idx, (key, W) in enumerate(weight_dict.items()):
    print(f"  [{idx+1:2d}/{len(weight_dict)}]  {key} … ", end="", flush=True)
    param_nodes[key] = build_param_node(W, RANK, LEVEL, MIN_DIM)
    print("done")
 
del weight_dict          # free the raw weight tensors
gc.collect()
print("\nAll param trees built.  Raw weights freed.")
 

Running offline SVD …
  [ 1/66]  layer0.q … done
  [ 2/66]  layer0.k … done
  [ 3/66]  layer0.v … done
  [ 4/66]  layer1.q … done
  [ 5/66]  layer1.k … done
  [ 6/66]  layer1.v … done
  [ 7/66]  layer2.q … done
  [ 8/66]  layer2.k … done
  [ 9/66]  layer2.v … done
  [10/66]  layer3.q … done
  [11/66]  layer3.k … done
  [12/66]  layer3.v … done
  [13/66]  layer4.q … done
  [14/66]  layer4.k … done
  [15/66]  layer4.v … done
  [16/66]  layer5.q … done
  [17/66]  layer5.k … done
  [18/66]  layer5.v … done
  [19/66]  layer6.q … done
  [20/66]  layer6.k … done
  [21/66]  layer6.v … done
  [22/66]  layer7.q … done
  [23/66]  layer7.k … done
  [24/66]  layer7.v … done
  [25/66]  layer8.q … done
  [26/66]  layer8.k … done
  [27/66]  layer8.v … done
  [28/66]  layer9.q … done
  [29/66]  layer9.k … done
  [30/66]  layer9.v … done
  [31/66]  layer10.q … done
  [32/66]  layer10.k … done
  [33/66]  layer10.v … done
  [34/66]  layer11.q … done
  [35/66]  layer11.k … done
  [36/66]  layer11.v … done


In [14]:
# ────────────────────────────────────────────────────────────
# CELL 6 — H-matrix nn.Modules  (USV throughout)
# ────────────────────────────────────────────────────────────
 
class USVBlock(nn.Module):
    """
    Single USV block:  out = (x @ V.T) * S  @ U.T
    Identical to ViT's TrainableUVBlock.
    """
    def __init__(self, U: torch.Tensor, S: torch.Tensor, V: torch.Tensor):
        super().__init__()
        self.U = nn.Parameter(U)    # (out, r)
        self.S = nn.Parameter(S)    # (r,)
        self.V = nn.Parameter(V)    # (r, in)
 
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return (x @ self.V.T) * self.S @ self.U.T
 
 
class HMatrixBlock(nn.Module):
    def __init__(self, node: dict):
        super().__init__()
        self.is_leaf = node["is_leaf"]
 
        if self.is_leaf:
            self.block = USVBlock(node["U"], node["S"], node["V"])
            return
 
        self.in_mid  = node["in_mid"]
        self.out_mid = node["out_mid"]
        self.A11 = HMatrixBlock(node["A11"])
        self.A22 = HMatrixBlock(node["A22"])
        self.A12 = USVBlock(node["A12"]["U"], node["A12"]["S"], node["A12"]["V"])
        self.A21 = USVBlock(node["A21"]["U"], node["A21"]["S"], node["A21"]["V"])
 
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if self.is_leaf:
            return self.block(x)
        x1 = x[..., :self.in_mid]
        x2 = x[..., self.in_mid:]
        y1 = self.A11(x1) + self.A12(x2)
        y2 = self.A21(x1) + self.A22(x2)
        return torch.cat([y1, y2], dim=-1)
 
 
class HierarchicalLinear(nn.Module):
    def __init__(self, node: dict):
        super().__init__()
        self.hblock = HMatrixBlock(node)
 
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        shape = x.shape
        return self.hblock(x.reshape(-1, shape[-1])).reshape(*shape[:-1], -1)
 
 
# Sanity check
_m = HierarchicalLinear(param_nodes["layer0.q"])
_p = sum(p.numel() for p in _m.parameters())
print(f"layer0.q  H-params={_p:,}  dense={2048**2:,}  ratio={_p/2048**2*100:.1f}%")
del _m

layer0.q  H-params=2,627,328  dense=4,194,304  ratio=62.6%


In [15]:
# ────────────────────────────────────────────────────────────
# CELL 7 — Load model across both T4s and swap projections
#
#  device_map="auto" splits the 22 transformer layers evenly:
#    GPU 0 → layers  0–10  (roughly)
#    GPU 1 → layers 11–21  (roughly)
#
#  After replacing q/k/v, each HierarchicalLinear is moved to
#  the same device as its parent layer — no cross-device ops.
#
#  Memory estimate (fp32):
#    Base model          ~4.4 GB
#    66 H-matrices       ~66 × 2.1M × 4B ≈ 555 MB extra
#    Total across 2 GPUs ≈ 2.5 GB per GPU   ← well within 16 GB
# ────────────────────────────────────────────────────────────
 
print("Loading model across both GPUs …")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float32,
    device_map="cuda:0",              # splits automatically across GPU 0 & 1
)
 
# Freeze everything
for p in model.parameters():
    p.requires_grad = False
 
# Replace q/k/v and unfreeze
for i, layer in enumerate(model.model.layers):
    attn = layer.self_attn
 
    # Detect which device this layer lives on
    layer_device = next(attn.parameters()).device
 
    attn.q_proj = HierarchicalLinear(param_nodes[f"layer{i}.q"]).to(layer_device)
    attn.k_proj = HierarchicalLinear(param_nodes[f"layer{i}.k"]).to(layer_device)
    attn.v_proj = HierarchicalLinear(param_nodes[f"layer{i}.v"]).to(layer_device)
 
    for proj in (attn.q_proj, attn.k_proj, attn.v_proj):
        for p in proj.parameters():
            p.requires_grad = True
 
    print(f"  layer {i:2d}  device={layer_device}  replaced & unfrozen")
 
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"\nTrainable : {trainable:,}  ({trainable/1e6:.2f} M)")
print(f"Total     : {total:,}  ({total/1e6:.2f} M)")
print(f"% trained : {100*trainable/total:.2f}%")
 
for i in range(torch.cuda.device_count()):
    free, tot = torch.cuda.mem_get_info(i)
    print(f"GPU {i} memory  used={( tot-free)/1e9:.2f}GB  free={free/1e9:.2f}GB")
 

Loading model across both GPUs …


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

  layer  0  device=cuda:0  replaced & unfrozen
  layer  1  device=cuda:0  replaced & unfrozen
  layer  2  device=cuda:0  replaced & unfrozen
  layer  3  device=cuda:0  replaced & unfrozen
  layer  4  device=cuda:0  replaced & unfrozen
  layer  5  device=cuda:0  replaced & unfrozen
  layer  6  device=cuda:0  replaced & unfrozen
  layer  7  device=cuda:0  replaced & unfrozen
  layer  8  device=cuda:0  replaced & unfrozen
  layer  9  device=cuda:0  replaced & unfrozen
  layer 10  device=cuda:0  replaced & unfrozen
  layer 11  device=cuda:0  replaced & unfrozen
  layer 12  device=cuda:0  replaced & unfrozen
  layer 13  device=cuda:0  replaced & unfrozen
  layer 14  device=cuda:0  replaced & unfrozen
  layer 15  device=cuda:0  replaced & unfrozen
  layer 16  device=cuda:0  replaced & unfrozen
  layer 17  device=cuda:0  replaced & unfrozen
  layer 18  device=cuda:0  replaced & unfrozen
  layer 19  device=cuda:0  replaced & unfrozen
  layer 20  device=cuda:0  replaced & unfrozen
  layer 21  d

In [16]:
# ────────────────────────────────────────────────────────────
# CELL 8 — WikiText-2 dataset
# ────────────────────────────────────────────────────────────
 
raw_datasets = load_dataset("wikitext", "wikitext-2-raw-v1")
 
tokenized = raw_datasets.map(
    lambda ex: tokenizer(ex["text"]),
    batched=True,
    remove_columns=["text"],
    desc="Tokenising",
)
 
def group_texts(examples):
    concat = {k: sum(examples[k], []) for k in examples.keys()}
    total  = (len(concat["input_ids"]) // BLOCK_SIZE) * BLOCK_SIZE
    result = {k: [v[i:i+BLOCK_SIZE] for i in range(0, total, BLOCK_SIZE)]
              for k, v in concat.items()}
    result["labels"] = result["input_ids"].copy()
    return result
 
lm_datasets = tokenized.map(group_texts, batched=True, desc="Grouping")
print(f"Train={len(lm_datasets['train'])}  "
      f"Val={len(lm_datasets['validation'])}  "
      f"Test={len(lm_datasets['test'])}")
 

README.md: 0.00B [00:00, ?B/s]

wikitext-2-raw-v1/test-00000-of-00001.pa(…):   0%|          | 0.00/733k [00:00<?, ?B/s]

wikitext-2-raw-v1/train-00000-of-00001.p(…):   0%|          | 0.00/6.36M [00:00<?, ?B/s]

wikitext-2-raw-v1/validation-00000-of-00(…):   0%|          | 0.00/657k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/4358 [00:00<?, ? examples/s]

...

Tokenising:   0%|          | 0/3760 [00:00<?, ? examples/s]

Grouping:   0%|          | 0/4358 [00:00<?, ? examples/s]

Grouping:   0%|          | 0/36718 [00:00<?, ? examples/s]

Grouping:   0%|          | 0/3760 [00:00<?, ? examples/s]

Train=5569  Val=581  Test=661


In [17]:
# DIAGNOSTIC — run before training

import torch

# Check where model parameters actually are
devices = set()
for name, p in model.named_parameters():
    devices.add(str(p.device))
print("Parameter devices:", devices)

# Check GPU utilization
for i in range(torch.cuda.device_count()):
    free, total = torch.cuda.mem_get_info(i)
    print(f"GPU {i}: used={(total-free)/1e9:.2f}GB  free={free/1e9:.2f}GB")

# Force a dummy forward pass and time it
model.eval()
dummy = torch.randint(0, 1000, (1, 64)).to("cuda:0")
with torch.no_grad():
    import time
    t = time.time()
    out = model(dummy)
    torch.cuda.synchronize()
    print(f"Forward pass time: {time.time()-t:.3f}s")

Parameter devices: {'cuda:0'}
GPU 0: used=9.26GB  free=6.38GB
Forward pass time: 0.553s


In [18]:
import os, json
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

OUTPUT_DIR = "/kaggle/working/hmatrix_tinyllama_wikitext"

model.gradient_checkpointing_enable()

training_args = TrainingArguments(
    output_dir                  = OUTPUT_DIR,
    num_train_epochs            = 3,
    per_device_train_batch_size = 2,
    per_device_eval_batch_size  = 2,
    gradient_accumulation_steps = 16,
    learning_rate               = 2e-4,
    weight_decay                = 0.01,
    lr_scheduler_type           = "cosine",
    warmup_steps                = 100,
    optim                       = "adamw_torch",
    fp16                        = True,
    logging_steps               = 50,
    eval_strategy               = "epoch",
    save_strategy               = "epoch",
    load_best_model_at_end      = True,
    metric_for_best_model       = "eval_loss",
    greater_is_better           = False,
    report_to                   = "none",
    dataloader_num_workers      = 2,
    use_cpu                     = False,
)

trainer = Trainer(
    model         = model,
    args          = training_args,
    train_dataset = lm_datasets["train"],
    eval_dataset  = lm_datasets["validation"],
    data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False),
)

print("Fine-tuning …")
trainer.train()
print("Done.")

# ── Save immediately after training ──────────────────────────
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"Model saved to {OUTPUT_DIR}")
print(os.listdir(OUTPUT_DIR))

# ── Zip and download ─────────────────────────────────────────
import shutil
zip_path = "/kaggle/working/hmatrix_model"
shutil.make_archive(zip_path, "zip", OUTPUT_DIR)
print(f"Zipped to {zip_path}.zip")

# Download link — click the file in Kaggle output panel
from IPython.display import FileLink
display(FileLink(f"hmatrix_model.zip"))

Fine-tuning …


Epoch,Training Loss,Validation Loss
1,2.104396,2.091834
2,1.973499,2.081215
3,1.845072,2.097275


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Done.


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved to /kaggle/working/hmatrix_tinyllama_wikitext
['tokenizer_config.json', 'model.safetensors', 'tokenizer.json', 'generation_config.json', 'checkpoint-525', 'checkpoint-350', 'config.json', 'checkpoint-175', 'training_args.bin']


OSError: [Errno 28] No space left on device

In [19]:
# CELL 10 — Test perplexity + accuracy

import math

results    = trainer.evaluate(eval_dataset=lm_datasets["test"])
perplexity = math.exp(results["eval_loss"])

# Next token prediction accuracy
model.eval()
correct = 0
total   = 0

with torch.no_grad():
    for batch in torch.utils.data.DataLoader(
        lm_datasets["test"],
        batch_size=1,
        collate_fn=DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)
    ):
        input_ids = batch["input_ids"].to("cuda:0")
        labels    = batch["labels"].to("cuda:0")

        outputs      = model(input_ids=input_ids)
        shift_logits = outputs.logits[..., :-1, :].contiguous()
        shift_labels = labels[..., 1:].contiguous()

        preds   = shift_logits.argmax(dim=-1)
        mask    = shift_labels != -100
        correct += (preds[mask] == shift_labels[mask]).sum().item()
        total   += mask.sum().item()

accuracy = 100.0 * correct / total

print("=" * 50)
print(f"Test loss  : {results['eval_loss']:.4f}")
print(f"Perplexity : {perplexity:.2f}")
print(f"Accuracy   : {accuracy:.2f}%")
print(f"Params     : {trainable/1e6:.2f}M  (rank={RANK}, level={LEVEL})")
print("=" * 50)

Test loss  : 2.0676
Perplexity : 7.91
Accuracy   : 55.55%
Params     : 83.79M  (rank=128, level=4)


In [ ]:
# CELL 11 — Save + generate

trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"Saved to {OUTPUT_DIR}")

model.eval()
prompt = "The history of artificial intelligence began"
inputs = tokenizer(prompt, return_tensors="pt").to("cuda:0")
with torch.no_grad():
    out = model.generate(
        **inputs,
        max_new_tokens = 80,
        temperature    = 0.7,
        do_sample      = True,
        top_p          = 0.9,
    )
print("\nGenerated:")
print(tokenizer.decode(out[0], skip_special_tokens=True))

In [2]:
from IPython.display import FileLink, display
display(FileLink("hmatrix_model.zip"))

/kaggle/working/hmatrix_model.zip

## Results

Accurary (without finetuning, Q only) = 49.66%

| Metric | H-Matrix (QKV) | LoRA (QKV) | H-Matrix (Q only,l=4,r=128) | LoRA (Q only) | H-Matrix (Q only,l=2,r=128) | LoRa (Q only)|
|---|---|---|---|---|---|---|
| **Trainable params** | 83.79M | 83.68M | 80.87M | 80.74M | 55.7 | 55.69
| **Epoch 1 train loss** | 2.1044 | 2.0718 | 2.1059 | 2.0825 | 2.1101| 2.0824
| **Epoch 2 train loss** | 1.9735 | 1.9512 | 1.9462 | 1.9320 | 1.9504| 1.9353
| **Epoch 3 train loss** | 1.8451 | 1.8218 | 1.7730 | 1.7507 | 1.7781| 1.7619
| **Epoch 1 val loss** | 2.0918 | 2.0613 | 2.0978 | 2.0757 | 2.1014| 2.0739
| **Epoch 2 val loss** | 2.0812 | 2.0706 | 2.0938 | 2.0780 | 2.097294| 2.0759
| **Epoch 3 val loss** | 2.0973 | 2.0961 | 2.1220 | 2.1120 | 2.1248| 2.1081
| **Test loss** | 2.0676 | 2.0478 | 2.0828 | 2.0878 | 2.0851| 2.0845
| **Test perplexity** | 7.91 | 7.75 | 8.03 | 8.07 | 8.05 | 8.04
| **Test accuracy** | 55.55% | 55.78% | 55.31% | 55.39% | 57.26%| 55.45%